# Activation coverage vs. reconstruction condition, and child geometry

Reproduces the analyses of Sections 2.2, 3 and 6.1 for a single parent.

In [ ]:
import torch
from transformer_lens import HookedTransformer

from tree_sae.analysis.activations import llm_activations, load_eval_tokens, sae_sparse_activations
from tree_sae.models.io import from_pretrained

torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device, dtype=torch.bfloat16)

In [ ]:
sae = from_pretrained("gpt2-small/tree_sae_2layer_l0_32", device=device).to(torch.bfloat16)
tokens = load_eval_tokens(10_000, 128, "gpt2")
acts = llm_activations(model, tokens, sae.cfg.hook_name)
indices, values = sae_sparse_activations(sae, acts, 128, device, verbose=True)
indices[values < 1e-3] = sae.cfg.d_sae

## Children by activation coverage and their probes (Fig. 1)

A child with perfect activation coverage can still be unrelated to its parent: its probe direction $d^*_c$ is far from the parent decoder vector.

In [ ]:
from tree_sae.analysis.probe_correlation import children_by_coverage, probe_report

parent = 1343
children, coverage = children_by_coverage(indices, parent, sae.cfg.d_sae, n_children=5)
for r in probe_report(sae, acts, indices, parent, children, device):
    print(
        f"child {r.child}: coverage {r.coverage:.2f}, parent similarity {r.parent_similarity:.3f} "
        f"(rank {r.parent_rank}), S_res {r.reconstruction_score:.3f}"
    )

## Geometry of the child subspace (Fig. 9)

In [ ]:
from tree_sae.analysis.geometry import plot_child_geometry

sae4 = from_pretrained("gpt2-small/tree_sae_4layer_l0_32", device=device).to(torch.bfloat16)
idx4, val4 = sae_sparse_activations(sae4, acts, 128, device)
idx4[val4 < 1e-3] = sae4.cfg.d_sae
ax, pca = plot_child_geometry(sae4, acts, idx4, parent=5)